In [ ]:
# Step 1: Clone the repository (main branch) and install dependencies.
!git clone -b main --single-branch https://github.com/HUBioDataLab/ContVAR.git /content/ContVAR
%cd /content/ContVAR

%pip install -q graphein MDAnalysis torch_geometric torchmetrics wandb biopython h5py
!apt-get -qq install dssp
%pip install -e .

In [ ]:
# Step 2: Mount Google Drive to access data files.
# Drive is only used for two large files (not code):€
#   - protein_triplets_data_9march.zip  (CIF structure files)
#   - embeddings_variable.h5            (precomputed ESM2 embeddings)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3: Login to Weights & Biases and set up the environment.
# - wandb.login() will prompt for your API key interactively.
# - setup_environment(..., prepare_dms_triplets_zip=False) skips unzipping here.
#   train_pipeline runs GO phase 0 first when enabled, then unzips DMS if needed,
#   then runs stage-2 DMS training.
import wandb
from contvar import setup_environment, train_pipeline, visualize_tsne

wandb.login(key="2becafa4dcb70173759a7b50ee5de92401c637c4")
env = setup_environment(prepare_dms_triplets_zip=False)
# env['device'], env['data_root'], env['embeddings_path'], env['data_zip']

In [ ]:
# Step 4: Phase 0 (GO) → DMS unzip if needed → Stage-2 DMS training.
# - Phase 0: GO semantic similarity pretraining (MF/BP/CC heads) using semantic_similarity TSVs
#   and prebuilt GO graph .pt files.
# - Stage 2 train: streaming semi-hard negative mining on train proteins.
# - Stage 2 val/test: exhaustive evaluation on held-out proteins from the fixed protein split JSON.
# - force=True  → generate all DMS graphs from scratch.
# - force=False → reuse previously processed graphs if available.

# Colab: TSV + prebuilt graphs can live on Drive.
# GO protein split JSON defaults to local_splits/phase0_protein_split_removed_graphless.json after clone.
# DMS protein split JSON defaults to local_splits/dms_protein_split.json after clone.
# If either split JSON lives somewhere else, override its path in config_overrides.
# NOTE: go_phase0_epochs is intentionally fixed to 200 in this notebook.
go_checkpoint = "/content/ContVAR/model_go_pretraining_best_loss.pt"

config_overrides = {
    # Load already-trained GO phase-0 checkpoint
    "go_phase0_init_checkpoint_path": go_checkpoint,

    # Skip GO phase-0 training
    "go_phase0_epochs": 0,
    "go_prebuilt_graph_root": None,

    # DMS output checkpoints
    "stage2_best_model_path": "model_best_loss.pt",
    "stage2_last_model_path": "model_last.pt",
    # Colab-friendly
    "num_workers": 2,
}


model, mapper, processed_dir = train_pipeline(
    config=config_overrides,
    force=False,
    data_root=env['data_root'],
    embeddings_path=env['embeddings_path'],
    device=env['device'],
    data_zip=env.get('data_zip'),
)

In [ ]:
# Step 5: Export DMS embeddings after training.
# The returned model is the best Stage-2 checkpoint after final evaluation.
import os
from contvar.config import ProjectConfig
from contvar.export_embeddings import export_all_embeddings

export_dir = "exports"
os.makedirs(export_dir, exist_ok=True)

export_cfg = ProjectConfig()
for key, value in config_overrides.items():
    if hasattr(export_cfg, key):
        setattr(export_cfg, key, value)

export_results = export_all_embeddings(
    model=model,
    cfg=export_cfg,
    device=env['device'],
    data_root=env['data_root'],
    embeddings_path=env['embeddings_path'],
    go_prebuilt_graph_root=export_cfg.go_prebuilt_graph_root,
    phase0_split_json_path=export_cfg.go_protein_split_json_path,
    dms_split_json_path=export_cfg.dms_protein_split_json_path,
    phase0_out_path=os.path.join(export_dir, "phase0_contvar_embeddings.h5"),
    dms_out_path=os.path.join(export_dir, "dms_variant_contvar_embeddings.h5"),
    batch_size=32,
    force_dms_reprocess=False,
    include_dms_anchors=False,
)
export_results

In [ ]:
# Step 6: Visualize learned embeddings using t-SNE.
# Plots comparing:
#   - Baseline (raw pooled node features, no projection) vs Projected (GNN output)
#   - Global (graph-level) vs Local (mutation-position) embeddings
# Each point is colored by label (benign=green, pathogenic=red, WT=gold).
import os
import torch

base_save_dir = "visualizations"
best_save_dir = os.path.join(base_save_dir, "best")
last_save_dir = os.path.join(base_save_dir, "last")

best_checkpoint = "model_best_loss.pt"
if os.path.exists(best_checkpoint):
    print(f"Loading best checkpoint: {best_checkpoint}")
    model.load_state_dict(torch.load(best_checkpoint, map_location=env['device']))

print("Generating t-SNE visualizations for best checkpoint...")
for split in ["val", "train"]:
    visualize_tsne(
        model=model, mapper=mapper, processed_dir=processed_dir,
        device=env['device'], save_dir=best_save_dir, split=split, n_proteins=10,
    )

last_checkpoint = "model_last.pt"
if os.path.exists(last_checkpoint):
    print(f"Loading last checkpoint: {last_checkpoint}")
    model.load_state_dict(torch.load(last_checkpoint, map_location=env['device']))
    print("Generating t-SNE visualizations for last checkpoint...")
    for split in ["val", "train"]:
        visualize_tsne(
            model=model, mapper=mapper, processed_dir=processed_dir,
            device=env['device'], save_dir=last_save_dir, split=split, n_proteins=10,
        )
else:
    print(f"Skipping last-checkpoint visualization; not found: {last_checkpoint}")

In [ ]:
import os
import shutil
from datetime import datetime

# Timestamped folder name: e.g. Protein_Model_Results_23Mar_1542
timestamp = datetime.now().strftime("%d%b_%H%M")
save_dir = f'/content/drive/MyDrive/Protein_Model_Results_{timestamp}'
os.makedirs(save_dir, exist_ok=True)
print(f"Saving to: {save_dir}")

files_to_save = [
    'model_phase0_best_loss.pt',
    'model_phase0_last.pt',
    'model_best_loss.pt',
    'model_last.pt',
    'local_splits/dms_protein_split.json',
    'local_splits/phase0_protein_split_removed_graphless.json',
    'exports',
    'visualizations',
]

for f in files_to_save:
    if os.path.isdir(f):
        dest = os.path.join(save_dir, os.path.basename(f))
        shutil.copytree(f, dest, dirs_exist_ok=True)
        print(f"Saved directory {f}")
    elif os.path.exists(f):
        shutil.copy(f, save_dir)
        print(f"Saved {f}")
    else:
        print(f"File not found: {f}")

print("All files safely transferred to Google Drive.")

In [ ]:
from google.colab import files

zip_name = f'/content/Protein_Model_Results_{timestamp}.zip'
!zip -r {zip_name} {save_dir}
files.download(zip_name)

In [ ]:
#from google.colab import runtime

#import time

#time.sleep(5)
# This will disconnect and delete the runtime
#runtime.unassign()